# Step 02. 종방향 운동 계획 (Longitudinal Motion Planning)

> **author**: Minwon Lee, Hanyang University
>
> **참고**: PythonRobotics (https://github.com/AtsushiSakai/PythonRobotics)
>
> **교육 주관**: Hanyang University, Automotive Intelligence Laboratory (AILAB)

Step 01에서 만든 cubic spline 경로는 **어디로** 갈지(형태 $x, y$)만 정해 줍니다.  
이 노트북에서는 그 경로 위에서 **얼마나 빠르게** 달릴지, 즉 각 지점의 속도(speed profile)를 정합니다.  
제한속도·방지턱·정지선 같은 제약과 경로의 곡률을 반영해 목표 속도를 만들고, 차가 실제로 낼 수 있도록 가·감속 한계로 다듬습니다.

**용어**
- **경로(path)**: 지나갈 선의 모양 (x, y) ← Step 01
- **궤적(trajectory)**: 경로 + 각 지점의 속도·시간 ← 이 노트북의 결과

## 학습 목표

1. 경로의 곡률 $\kappa$ 계산 (회전반경 $R = 1/\kappa$ 와의 역수 관계 이해)
2. 목표 속도 프로파일 만들기
   - 2.1. 행동 기준 속도 $v_{ref}$ (제한속도 60 / 중간 30 구간 · 방지턱 · 정지선)
   - 2.2. 곡률 제한 속도 $v_{\kappa}$
   - 2.3. 목표 속도 $v_f = \min(v_{ref},\, v_{\kappa})$
3. 가·감속 한계로 속도 프로파일 스무싱 (forward / backward)
4. 결과 검증(가속도 한계) · 시간 파라미터화 $v(t),\, s(t)$ · 주행 애니메이션

## 0. Setup & 경로 준비

step01 에서 배운 cubic spline 으로 새 S자 도로를 만듭니다. step01 은 $y$ 가 $x$ 의 스플라인(1차원)이었다면, 여기서는 경로를 따라 잰 거리 $s$ (arc length) 를 기준으로 $x(s),\, y(s)$ 를 각각 스플라인으로 만드는 2차원 버전(`CalcSplineCourse`)입니다. 덕분에 경로를 등간격 $\Delta s$ 로 샘플링할 수 있고, 각 점의 진행 방향(yaw)과 곡률도 함께 얻습니다.

시나리오 파라미터(제한속도·방지턱·정지선·가속 한계)를 한곳에 모으고, `ShowScene`으로 장면을 먼저 확인합니다. 자차(ego)는 시작점에서 이미 제한속도 60 km/h 로 주행 중이라고 가정합니다. (단, a_min 을 작게 바꾸면 첫 커브 전에 감속을 끝내기 위해 출발 속도부터 낮아집니다. 약하게만 감속할 수 있다는 뜻입니다.)

In [ ]:
%matplotlib inline
import sys, os
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('../utils/02_trajectory_planning'))

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML

from step01_cubic_spline_path import CalcSplineCourse
from step02_longitudinal_motion_planning import (
    DrawScene, ShowScene, IndexAtS,
    PlotCurvature, PlotProfiles, PlotSmoothing,
    PlotValidation, PlotTimeParameterization, AnimateDrive,
)
print('ready')

In [ ]:
# step01 에서 배운 cubic spline 으로 만든 새 S자 도로.
# 직선 - 좌커브 - 직선 - 우커브 - 직선 으로 이어지는 가로로 긴 경로.
# 등간격 ds 로 arc-length(경로를 따라 잰 거리) 매개변수화
waypoint_x = [0.0, 19.7, 39.5, 59.5, 77.9, 94.5, 110.5, 126.8,
              145.3, 165.0, 185.0, 204.8, 224.5, 244.5, 264.3, 284.3]   # waypoint 들의 x 좌표
waypoint_y = [0.0,  0.0,  0.0,  1.0,  7.6, 18.8,  30.3,  41.5,
               48.7, 49.8, 49.8, 49.8, 49.8, 49.8, 49.8, 49.8]           # waypoint 들의 y 좌표
ds = 0.5   # ds: 샘플 간격 [m] (arc length 를 따라 점을 얼마나 촘촘히 찍을지)

# x, y      : 경로 점 좌표 / yaw : 각 점의 진행 방향(heading) [rad]
# kappa_ref : 라이브러리가 계산한 참고용 곡률 (§1 에서 우리 구현과 비교)
# s         : 시작점부터의 경로 거리(arc length) [m]
x, y, yaw, kappa_ref, s = map(np.array, CalcSplineCourse(waypoint_x, waypoint_y, ds=ds))
num_samples = len(s)   # num_samples: 경로 샘플 점 개수
print(f'경로 샘플 수 = {num_samples},  전체 길이 = {s[-1]:.1f} m,  ds = {ds} m')

In [ ]:
# ----- 단위 변환 헬퍼 (내부 계산은 m/s, 표시는 km/h) -----
def KphToMs(v_kph):   return v_kph / 3.6   # km/h -> m/s
def MsToKph(v_ms):    return v_ms * 3.6    # m/s  -> km/h

# ----- 시나리오 파라미터 -----
v_limit = KphToMs(60)         # v_limit: 기본 제한속도 60 km/h (m/s 로 저장)
v_limit30 = KphToMs(30)       # v_limit30: 중간 구간 제한속도 30 km/h (m/s)
limit30_start, limit30_end = 178.0, 212.0  # 30 km/h 구간의 시작/끝 거리 [m]

v_bump = KphToMs(10)          # v_bump: 방지턱 통과 제한 10 km/h (m/s)
bump_start, bump_end = 260.0, 265.0        # 방지턱 구간의 시작/끝 거리 [m]

s_stop = 288.0                # s_stop: 정지선/적색신호 위치 [m] (이후 속도 0)

a_lat_max = 2.0               # a_lat_max: 곡률 제한용 횡(가로)방향 가속도 한계 [m/s^2]
a_max     = 1.5               # a_max: 종(앞뒤)방향 가속 한계 [m/s^2]
a_min     = -2.5              # a_min: 종방향 감속 한계 [m/s^2] (음수)

# ----- 장면(scene): DrawScene 가 쓰는 정보 묶음 -----
scene = dict(s=s, x=x, y=y, yaw=yaw,
             v_limit_kph=60, limit30_kph=30,
             limit30_start=limit30_start, limit30_end=limit30_end,
             v_bump_kph=10, bump_start=bump_start, bump_end=bump_end,
             s_stop=s_stop, ds=ds, a_lat_max=a_lat_max)
print('scenario ready')

In [ ]:
# 시나리오 한눈에 보기 (제한속도 60/30 표지판 / 방지턱+경고 / 정지선+신호등 / 자차)
ShowScene(scene, v_kph=None, title='Longitudinal planning scenario')

## 1. 곡률 $\kappa$ 계산

각 점에서 경로가 휘는 정도가 곡률 $\kappa$ 입니다. 회전반경 $R$ 과는 역수 관계라($R = 1/\kappa$) 곡률이 클수록 급커브입니다.

평면 곡선의 곡률은 1·2차 미분으로 구합니다. (arc-length 매개변수라 분모 $(x'^2+y'^2)$ 는 약 1이지만, 일반식으로 구현합니다.)

$$\kappa = \frac{|x'\,y'' - y'\,x''|}{(x'^2 + y'^2)^{3/2}}$$

곡률의 부호는 좌/우 회전 방향을 나타내는데, 속도 제한에는 크기만 필요하므로 절댓값을 씁니다. S자 경로라 좌·우 두 군데에서 곡률이 커집니다. 직선처럼 보이는 구간에도 스플라인이 waypoint 를 부드럽게 지나느라 아주 작은 곡률이 남는데, 속도에 영향을 줄 정도는 아닙니다.

곡률 그래프가 계단처럼 튀지 않고 이어지는 것은 step01 에서 본 **C2 연속**(2차 미분 연속) 덕분입니다. 곡률은 2차 미분으로 계산하므로, C1 까지만 이어진 곡선이었다면 접점에서 곡률이 계단처럼 튑니다.

In [ ]:
# 경로 거리 s 에 대한 x, y 의 미분을 np.gradient 로 구한다 (곡률 공식의 재료).
# np.gradient(값, s): 이웃 샘플의 차이로 미분을 근사하는 수치 미분
# dx  = dx/ds,     dy  = dy/ds      : 1차 미분 (경로의 접선 방향)
# ddx = d^2x/ds^2, ddy = d^2y/ds^2  : 2차 미분 (경로가 휘어지는 정도)
dx  = np.gradient(x, s)
dy  = np.gradient(y, s)
ddx = np.gradient(dx, s)
ddy = np.gradient(dy, s)

def Curvature(dx, dy, ddx, ddy):
    ##### TODO 1 #####
    # 곡률 계산하기: κ = |x'·y'' - y'·x''| / (x'² + y'²)^(3/2)   (섹션 1 공식, x'=dx, y'=dy, x''=ddx, y''=ddy)
    # 사용함수: np.abs(np.ndarray)        / 기능: 원소별 절댓값
    # 사용함수: np.power(base, exponent)  / 기능: 원소별 거듭제곱 (base ** exponent 와 동일)
    # 주의: 거듭제곱은 ** (^ 는 다른 연산 -> 에러) / 지수 3/2 는 1.5 또는 (3/2) 로 괄호 필수 (x**3/2 = x³ ÷ 2)
    #################

    kappa = ...

    #################
    return kappa

kappa = Curvature(dx, dy, ddx, ddy)       # kappa: 각 점의 곡률 κ [1/m] (클수록 급한 커브)

# 라이브러리가 준 참고 곡률(kappa_ref, 부호 있음)과 크기 비교 -- 구현 확인용
# (np.gradient 는 수치 미분이라 라이브러리의 해석적 값과 아주 작은 차이가 남)
diff = np.abs(kappa - np.abs(kappa_ref)).max()
print(f'라이브러리 곡률과 최대 차이 = {diff:.2e}  (정상: 1e-3 보다 훨씬 작음, 약 2e-4)')
assert diff < 1e-3, 'TODO 1 확인 - 곡률 오답: 분모 지수 3/2 확인 -> (…) ** 1.5 또는 np.power(…, 1.5)'

kappa = np.clip(kappa, 1e-4, None)        # 작은 kappa 하한 (0 나눗셈/발산 방지)

# 경로 거리 s 를 따라 곡률 κ 를 그린다 (κ 큰 곳 = 급커브, S자라 봉우리 2개)
PlotCurvature(scene, kappa)

## 2.1. 행동 기준 속도 $v_{ref}$  (슬라이드 26)

가장 기본이 되는 기준 속도입니다. "행동(behavior)"은 상위 판단 단계가 정해 주는 도로 규칙(제한속도·방지턱·정지선)을 말합니다. 기본은 제한속도이고, 규칙에 따라 **아래로** 깎습니다.

- 기본: $v_{ref} = v_{limit}$ (60 km/h)
- 30 km/h 구간 $s \in [s_{30start},\, s_{30end}]$: $v_{ref} = 30$ km/h
- 방지턱 구간 $s \in [s_{bstart},\, s_{bend}]$: $v_{ref} = v_{bump}$ (10 km/h)
- 정지선 이후 $s \ge s_{stop}$: $v_{ref} = 0$

In [ ]:
# v_ref: 행동 기준 속도. 전 구간을 기본 제한속도(60)로 초기화한 뒤 규칙에 따라 아래로 깎음.
v_ref = np.full(num_samples, v_limit)   # np.full(길이, 값): 그 값으로 가득 채운 배열 생성

##### TODO 2 #####
# 1) 30 km/h 구간 [limit30_start, limit30_end] 은 v_limit30 을 상한으로 적용
# 2) 방지턱 구간 [bump_start, bump_end] 은 v_bump 를 상한으로 적용
# 3) 정지선 이후 (s >= s_stop) 는 0
# 사용기법: boolean mask indexing -- v_ref[조건] = 값  (조건이 True 인 원소에만 대입)
#          조건은 (s >= a) & (s <= b) 형태, & 는 원소별 AND
#################

v_ref[...] = ...
v_ref[...] = ...
v_ref[...] = ...

#################

In [ ]:
PlotProfiles(scene, [{'v': v_ref, 'label': 'v_ref', 'color': 'tab:blue'}],
              '2.1: behavioral reference velocity  v_ref')

## 2.2. 곡률 제한 속도 $v_{\kappa}$  (슬라이드 28)

커브를 빠르게 돌면 원심력(횡가속도)이 커집니다. 횡가속도를 한계 $a_{lat,max}$ 이내로 두면 곡률이 큰 곳의 최대 속도가 정해집니다.

$$a_{lat} = v^2 \kappa \le a_{lat,max} \quad\Rightarrow\quad v_{\kappa} \le \sqrt{\frac{a_{lat,max}}{\kappa}}$$

곡률이 매우 작은(거의 직선) 구간은 $v_{\kappa}$ 가 발산하므로 제한속도를 상한으로 둡니다(`np.minimum`). S자 두 커브에서 $v_{\kappa}$ 가 낮아집니다.

In [ ]:
##### TODO 3 #####
# v_kappa: 곡률 제한 속도. 횡가속도 a_lat = v^2 * kappa 를 한계 안에 두면 커브별 최대 속도가 정해짐.
# 사용함수: np.sqrt(np.ndarray)  / 기능: 원소별 제곱근
#################

v_kappa = ...

#################

v_kappa = np.minimum(v_kappa, v_limit)   # 거의 직선 구간 발산 방지: 제한속도를 상한으로

In [ ]:
PlotProfiles(scene, [{'v': v_kappa, 'label': 'v_kappa', 'color': 'tab:green'}],
              '2.2: curvature speed limit  v_kappa')

## 2.3. 목표 속도 $v_f$  (슬라이드 29)

두 제약 중 더 엄격한(낮은) 값을 각 점마다 고릅니다.

$$v_f = \min(v_{ref},\; v_{\kappa})$$

이 $v_f$ 는 아직 "지켜야 할 상한"일 뿐, 가·감속 한계는 반영되지 않아 톱니처럼 급변할 수 있습니다. 다음 3.1~3.2에서 다듬어 최종 속도를 만듭니다.

> 이후 그림에서 굵은 파란 실선은 항상 그 단계의 결과이고, 회색·주황 점선은 이전 단계의 결과입니다.

In [ ]:
# v_f: 목표 속도. 두 제약(v_ref, v_kappa) 중 더 엄격한(낮은) 값을 각 점마다 선택.
#      (아직 "지켜야 할 상한"일 뿐, 최종 속도는 3.1~3.2 smoothing 후에 나옴)
##### TODO 4 #####
# 사용함수: np.minimum(a, b)  / 기능: 두 배열의 원소별 최솟값
#################

v_f = ...

#################

In [ ]:
PlotProfiles(scene, [
    {'v': v_ref,   'label': 'v_ref',     'color': 'gray',      'ls': ':',  'lw': 1.8},
    {'v': v_kappa, 'label': 'v_kappa',   'color': 'tab:green', 'ls': '--', 'lw': 1.8},
    {'v': v_f,     'label': 'v_f = min', 'color': 'tab:blue',  'lw': 2.8},
], '2.3 target speed  v_f = min(v_ref, v_kappa)')

## 3.1. Forward smoothing (가속 한계)  (슬라이드 30)

$v_f$ 는 저속 구간(30 구간·방지턱 같은 slow zone)을 빠져나올 때 순간적으로 속도가 튈 수 있습니다. 실제 차는 가속 한계 $a_{max}$ 이상으로 빨라질 수 없습니다. 앞으로 한 칸씩 가며, 직전 속도에서 $a_{max}$ 로 가속해 도달 가능한 값으로 상한을 둡니다.

$$\text{if } a = \frac{v_i^2 - v_{i-1}^2}{2\Delta s} > a_{max}: \quad v_i \leftarrow \sqrt{v_{i-1}^2 + 2\,a_{max}\,\Delta s}$$

이 식은 등가속도 공식 $v^2 = v_0^2 + 2a\Delta s$ 에서 나온 것입니다. 코드에서는 조건문 대신 `min` 을 쓰는데, 한계를 넘을 때만 깎이므로 위 조건문과 같은 효과입니다.

In [ ]:
# v_forward: 가속 한계를 반영한 속도 (v_f 를 앞에서부터 훑으며 상한을 둠). forward = 앞으로 진행
v_forward = v_f.copy()
for i in range(1, num_samples):
    ##### TODO 5 #####
    # 가속 한계: v_forward[i] 는 v_forward[i-1] 에서 a_max 로 가속해 도달 가능한 값을 못 넘음
    # (위 공식을 참고해 v_forward[i] 를 갱신하세요)
    # 사용함수: min(a, b)   / 기능: 두 값 중 작은 값 (파이썬 내장)
    # 사용함수: np.sqrt(x)  / 기능: 제곱근
    #################

    v_forward[i] = ...

    #################

In [ ]:
PlotSmoothing(scene, v_f, v_forward, 'v_f (before)', 'after forward',
               '3.1: forward smoothing (acceleration limit)')

## 3.2. Backward smoothing (감속 한계)  (슬라이드 31)

이번엔 저속 구간에 **미리** 감속해 들어가도록, 뒤에서 앞으로 가며 제동 한계 $a_{min}\,(<0)$ 을 반영합니다. 가속도 $a = \frac{v_{i+1}^2 - v_i^2}{2\Delta s}$ 가 제동 한계보다 급하면, 앞 점 $v_{i+1}$ 까지 감속으로 도달 가능한 값으로 현재 속도의 상한을 둡니다.

$$\text{if } a < a_{min}: \quad v_i \leftarrow \sqrt{v_{i+1}^2 - 2\,a_{min}\,\Delta s} \quad (a_{min}<0 \text{ 이라 근호 안은 } +)$$

forward(가속) + backward(감속) 를 모두 거치면 가·감속이 물리 한계 안에 드는 실현 가능한 프로파일이 됩니다.

In [ ]:
# v_final: 감속 한계까지 반영한 최종 속도 (v_forward 를 뒤에서부터 훑으며 상한을 둠)
v_final = v_forward.copy()
# 뒤에서 두 번째 점부터 0번 점까지 역순으로 (마지막 점은 비교할 v[i+1] 이 없음)
for i in range(num_samples - 2, -1, -1):
    ##### TODO 6 #####
    # 감속(제동) 한계: 앞 점 v_final[i+1] 까지 a_min 으로 줄일 수 있어야 함 (a_min < 0 이라 근호 안은 +)
    # (위 공식을 참고해 v_final[i] 를 갱신하세요)
    # 사용함수: min(a, b)   / 기능: 두 값 중 작은 값 (파이썬 내장)
    # 사용함수: np.sqrt(x)  / 기능: 제곱근
    #################

    v_final[i] = ...

    #################

In [ ]:
PlotSmoothing(scene, v_forward, v_final, 'forward only', 'forward + backward',
               '3.2: backward smoothing (deceleration limit)')

## 4.1. 결과 검증 & 시간 파라미터화

완성된 프로파일이 정말 가·감속 한계 $[a_{min},\, a_{max}]$ 안에 드는지 확인하고, 거리($s$) 기준 속도를 **시간($t$) 기준**으로 바꿉니다. 한 칸을 평균 속도로 지난다고 보면

$$t_{i+1} = t_i + \frac{\Delta s}{(v_i + v_{i+1})/2}$$

정지선 이후는 속도가 0이라 시간이 무한대가 되므로, 정지선까지만 적분합니다.

In [ ]:
# (1) 가속도 프로파일 a(s): 인접 두 점의 속도 차이로 계산
accel = np.zeros(num_samples)      # accel: 각 점의 종방향 가속도 [m/s^2]
for i in range(num_samples - 1):
    ##### TODO 7 #####
    # accel[i] 계산하기: v² = v0² + 2·a·ds 를 a 에 대해 풀기 (3.1/3.2 와 같은 관계)
    #   - v0 = v_final[i], v = v_final[i + 1]
    # (라이브러리 함수 없이 사칙연산만 사용)
    # 주의: 분모는 괄호로 (2 * ds) -> '/ 2 * ds' 는 '÷ 2 후 × ds' 로 계산됨
    #################

    accel[i] = ...

    #################

print(f'accel 범위: [{accel.min():.2f}, {accel.max():.2f}] m/s^2  (한계 [{a_min}, {a_max}])')
# 확인: smoothing 이 맞으면 가·감속 한계에 딱 닿는 구간이 있음 (최소 = a_min, 최대 = a_max)
assert abs(accel.min() - a_min) < 0.05 and abs(accel.max() - a_max) < 0.05, \
    ('TODO 7 (accel) 또는 TODO 5·6 (smoothing) 확인: 정상이면 최소 -2.50 / 최대 1.50 에 닿음 '
     '-> accel 분모 (2 * ds) 괄호, 3.1/3.2 의 v_forward[i-1] / v_final[i+1] 확인')

# (2) 시간 파라미터화 -- 정지선까지만 적분
stop_index = IndexAtS(scene, s_stop)   # stop_index: 정지선에 가장 가까운 샘플 인덱스
s_to_stop = s[:stop_index + 1]           # s_to_stop: 정지선까지의 경로 거리 배열
v_to_stop = v_final[:stop_index + 1]     # v_to_stop: 정지선까지의 속도 배열
t = np.zeros(len(v_to_stop))             # t: 각 점에 도달하는 시간 [s]
for i in range(len(v_to_stop) - 1):
    v_average = (v_to_stop[i] + v_to_stop[i+1]) / 2   # v_average: 두 점 사이 구간의 평균 속도
    ##### TODO 8 #####
    # t[i+1] 계산하기: 위 공식, 0 나눗셈 방지로 v_average 에 작은 하한
    # 사용함수: max(a, b)  / 기능: 두 값 중 큰 값 (파이썬 내장, 하한 적용에 사용)
    #################

    t[i+1] = ...

    #################

print(f'총 주행 시간 = {t[-1]:.1f} s,  정지선까지 거리 = {s_to_stop[-1]:.1f} m')
# 확인: 기본 시나리오면 총 주행 시간 31.1 s (30 구간·방지턱·정지선 중 하나라도 빠지면 26.9 ~ 29.5 s)
print('✅ 총 주행 시간 확인 통과' if abs(t[-1] - 31.1) < 0.1
      else '❌ 기대값 31.1 s -> TODO 2 (v_ref 규칙 3개) 또는 TODO 8 (t 적분) 확인 (시나리오 값을 바꿨다면 무시)')

In [ ]:
PlotValidation(scene, v_final, accel, a_min, a_max)
PlotTimeParameterization(t, s_to_stop, v_to_stop)

## 4.2. 주행 애니메이션

최종 $v(t),\, s(t)$ 를 따라 자차가 경로를 달리는 모습입니다. 위 그림의 자차 아이콘이 움직이고, 아래 속도 그래프의 빨간 커서가 현재 위치를 가리킵니다. 곡선·30 구간·방지턱·정지선에서 속도가 떨어지고 완만한 구간에서 회복되는 것을 확인하세요.

(무거운 셀이므로 맨 끝에 둡니다. 실습 중에는 건너뛰고 마지막에 실행하세요. 재생 버튼/슬라이더로 제어합니다.)

In [ ]:
# 일정 시간 간격으로 t 를 균일 재샘플 -> 위치/속도 보간 (120 프레임)
n_frames = 120
t_uniform     = np.linspace(0, t[-1], n_frames)              # 균일한 시간 격자
s_uniform     = np.interp(t_uniform, t, s_to_stop)           # 각 시간에서의 경로 거리
v_uniform_kph = MsToKph(np.interp(t_uniform, t, v_to_stop))  # 각 시간에서의 속도 [km/h]

anim = AnimateDrive(scene, t_uniform, s_uniform, v_uniform_kph, interval=60)
HTML(anim.to_jshtml())